# Part 4 — Text and typography (VS Code notebook)

This notebook is for **inspecting and reusing** QUONITRON text components in VS Code.

## Setup

1. Open this repository folder in VS Code.
2. Select the project interpreter (the `.venv` or environment where you installed `quonitron-manim-master-library`).
3. Install the Python and Jupyter extensions if you want cell-by-cell execution.
4. Run the cells in order.

## What this environment can and cannot do

- **Wrapping, catalog search, and timeline checks** run without Cairo.
- **Notebook execution** uses the optional `notebook` extra (`ipykernel`, `nbclient`, and `nbformat`); CI executes the code cells.
- This notebook does not embed a Manim scene. For an actual animation preview, use the CLI on a host with Cairo, Pango, and FFmpeg:

```bash
manim -ql examples/text_typography_demo.py AssignmentInType
```

Then open the MP4 under `media/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "quonitron_manim").is_dir():
    # VS Code often starts the kernel in the repo root; if you opened notebooks/, go up.
    if (ROOT.parent / "src" / "quonitron_manim").is_dir():
        ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from quonitron_manim.components.text.wrap import wrap_plain_text, highlight_parts
from quonitron_manim.core.catalog import search_catalog
from quonitron_manim.design.layout import content_area
from quonitron_manim.design.theme import get_theme

print("package root", ROOT)
print("wrap:", wrap_plain_text("A variable is a name that points at a value.", 24))
print("highlight:", highlight_parts("score is the name.", "score"))
theme = get_theme()
box = content_area(theme.frame_width, theme.frame_height, theme.spacing.safe_margin)
print("content area bottom", round(box.bottom, 3), "(captions sit below this)")
print("search:", [h["id"] for h in search_catalog("paragraph", root=ROOT)[:6]])

## Reuse a component in a scene

```python
from quonitron_manim.components.text import CodeLine, HighlightedPhrase
from quonitron_manim.core.base_scene import QuonitronScene
from quonitron_manim.motion.appearance import WriteMotion

class Mini(QuonitronScene):
    def construct(self):
        line = CodeLine(theme=self.theme, text="score = 10").get()
        self.play(WriteMotion(theme=self.theme, run_time=1.2).animate(line))
```

Change copy with the `text` / `phrase` parameters. Change timing in `examples/text_timeline.json` (`timing_status: estimated` until you set `manual`). Export SRT/VTT:

```bash
python scripts/timeline_report.py examples/text_timeline.json --out examples/text_timeline_report.md
```

In [ ]:
from quonitron_manim.core.manim_compat import cairo_usable, manimpango_usable
from quonitron_manim.timeline.models import load_timeline
from quonitron_manim.timeline.validate import validate_timeline
from quonitron_manim.components.text import ALL_TEXT_COMPONENTS
from quonitron_manim.motion import ALL_MOTION_RECIPES
from quonitron_manim.transitions import ALL_TRANSITIONS

tl = load_timeline(ROOT / "examples" / "text_timeline.json")
known = [c.component_id for c in ALL_MOTION_RECIPES + ALL_TRANSITIONS + ALL_TEXT_COMPONENTS]
report = validate_timeline(tl, known_motion_ids=known)
print("timeline", tl.scene.scene_id, "cues", len(tl.cues), "ok", report.ok)
print("cairo", cairo_usable())
print("pango", manimpango_usable())
print("To render: manim -ql examples/text_typography_demo.py AssignmentInType")

## Troubleshooting

| Symptom | What to check |
| --- | --- |
| `ModuleNotFoundError: quonitron_manim` | Kernel cwd is the repo root; `sys.path` includes `src/`, or `pip install -e .` |
| `%%manim` is unknown | This notebook does not load the Manim IPython extension. Use the CLI command above. |
| Text looks like boxes / crash in `Text()` | Install real Pango fonts (`manimpango.list_fonts()` must be non-empty). |
| Equation falls back to plain text | Install LaTeX if you need `MathTex`. |
| Captions cover the lesson | Use `content_area()`; do not place educational copy in the bottom caption band. |